# 02 — Test Evaluation

## 1. Configuration

In [ ]:
import torch
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Paths (adjust to your Google Drive)
DATA_DIR  = Path('/content/drive/MyDrive/USK-Coffee')
LOCAL_DIR = Path('/content/USK-Coffee')
OUT_ROOT  = Path('/content/drive/MyDrive/coffee-two-stage/models')
TEST_OUT  = Path('/content/drive/MyDrive/coffee-two-stage/test_results')

# Data
CLASSES = ['defect', 'longberry', 'peaberry', 'premium']
TEST_COUNT = 400
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
BATCH_SIZE = 32
NUM_WORKERS = 2

# Experiment grid: 4class = flat, 3class = stage 1, 2class = stage 2
ARCHS   = ['resnet50', 'efficientnet_b0']
SCHEMES = ['4class', '3class', '2class']
SEEDS   = [0, 1, 2]
NUM_CLASSES = {'4class': 4, '3class': 3, '2class': 2}

# McNemar significance level
ALPHA = 0.05

def run_dir(arch, scheme, seed):
    return OUT_ROOT / arch / scheme / f'seed{seed}'

TEST_OUT.mkdir(parents=True, exist_ok=True)

## 2. Check models and test images

In [ ]:
EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def list_images(folder):
    return sorted(f for f in folder.iterdir() if f.suffix.lower() in EXTS)

# Weights are not needed if predictions are saved
missing = [run_dir(a, s, sd) for a in ARCHS for s in SCHEMES for sd in SEEDS
           if not (run_dir(a, s, sd) / 'final_model.pt').exists()
           and not (run_dir(a, s, sd) / 'test_predictions.npz').exists()]
assert not missing, f'Missing trained models: {missing}'

for c in CLASSES:
    n = len(list_images(DATA_DIR / 'test' / c))
    assert n == TEST_COUNT, f'test/{c}: {n} images, expected {TEST_COUNT}'
print('Models or saved predictions found for all 18 runs, test image counts OK')

## 3. Copy test data to the Colab disk

In [ ]:
import shutil, time

t0 = time.time()
dst = LOCAL_DIR / 'test'
ok = dst.exists() and all(len(list_images(dst / c)) == TEST_COUNT for c in CLASSES if (dst / c).exists())
if not ok:
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(DATA_DIR / 'test', dst)
for c in CLASSES:
    assert len(list_images(dst / c)) == TEST_COUNT, f'test/{c} incomplete'
print(f'done in {time.time() - t0:.0f}s')

## 4. Dataset and transforms

In [ ]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

LABELS_4CLASS = {'defect': 0, 'longberry': 1, 'peaberry': 2, 'premium': 3}

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CoffeeDataset(Dataset):
    def __init__(self, split_dir, transform):
        self.samples = [(f, label) for c, label in LABELS_4CLASS.items()
                        for f in list_images(split_dir / c)]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        with Image.open(path) as img:
            return self.transform(img.convert('RGB')), label

test_ds = CoffeeDataset(LOCAL_DIR / 'test', eval_tf)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(len(test_ds), 'test images')

## 5. Test predictions

In [ ]:
import torch.nn as nn
from torchvision import models

device = torch.device('cuda')

def build_model(arch, num_classes):
    if arch == 'resnet50':
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(arch)
    return m

@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        probs.append(torch.softmax(model(x.to(device)), 1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

test_paths = np.array([str(p.relative_to(LOCAL_DIR)) for p, _ in test_ds.samples])

# Test set is predicted once; saved predictions are reused
for arch in ARCHS:
    for scheme in SCHEMES:
        for seed in SEEDS:
            d = run_dir(arch, scheme, seed)
            f = d / 'test_predictions.npz'
            if f.exists():
                print(f'skip (already predicted): {d}')
                continue
            model = build_model(arch, NUM_CLASSES[scheme])
            model.load_state_dict(torch.load(d / 'final_model.pt', map_location='cpu'))
            prob, y_true = predict(model.to(device), test_loader)
            np.savez(f, prob=prob, y_true=y_true, paths=test_paths)
            print(f'predicted: {d}')

## 6. Metrics per seed

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

def load_preds(arch, scheme, seed):
    z = np.load(run_dir(arch, scheme, seed) / 'test_predictions.npz')
    assert (z['paths'] == test_paths).all(), 'image order mismatch'
    return z

# Merged-class images go to stage 2; others keep the stage-1 label
def hierarchical_predict(s1_prob, s2_prob):
    # Stage 1: 0 defect, 1 longberry, 2 merged
    s1 = s1_prob.argmax(1)
    # Stage 2: 0 peaberry, 1 premium
    s2 = s2_prob.argmax(1)
    # Back to the four classes
    return np.where(s1 == 2, 2 + s2, s1)

def metrics_row(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(4), zero_division=0)
    errors = (y_true != y_pred).sum()
    pb_to_pr = ((y_true == 2) & (y_pred == 3)).sum()
    pr_to_pb = ((y_true == 3) & (y_pred == 2)).sum()
    row = {'accuracy': (y_true == y_pred).mean(), 'macro_f1': f1.mean(), 'errors': errors,
           'peaberry->premium': pb_to_pr, 'premium->peaberry': pr_to_pb,
           'defect->premium': ((y_true == 0) & (y_pred == 3)).sum(),
           # Errors outside the pair
           'other_errors': errors - pb_to_pr - pr_to_pb,
           'pair_share_of_errors': (pb_to_pr + pr_to_pb) / max(errors, 1)}
    for k, c in enumerate(CLASSES):
        row.update({f'P_{c}': p[k], f'R_{c}': r[k], f'F1_{c}': f1[k]})
    return row

rows, cms, preds = [], {}, {}
for arch in ARCHS:
    for seed in SEEDS:
        flat, s1, s2 = (load_preds(arch, sc, seed) for sc in SCHEMES)
        y = flat['y_true']
        pred_flat = flat['prob'].argmax(1)
        pred_hier = hierarchical_predict(s1['prob'], s2['prob'])
        preds[(arch, seed)] = (y, pred_flat, pred_hier)

        rows.append({'arch': arch, 'seed': seed, 'model': 'flat', **metrics_row(y, pred_flat)})
        s1_pred = s1['prob'].argmax(1)
        pair = np.isin(y, [2, 3])
        extra = {
            # Stage-1 accuracy
            'stage1_acc': (np.where(y >= 2, 2, y) == s1_pred).mean(),
            'stage2_acc_on_pair': (s2['prob'].argmax(1)[pair] == y[pair] - 2).mean(),
            # Pair images lost at stage 1
            'pair_lost_at_stage1': (pair & (s1_pred != 2)).sum(),
            # Defect/longberry sent to stage 2
            'other_routed_to_stage2': (~pair & (s1_pred == 2)).sum(),
        }
        rows.append({'arch': arch, 'seed': seed, 'model': 'hierarchical', **metrics_row(y, pred_hier), **extra})
        cms[(arch, 'flat', seed)] = confusion_matrix(y, pred_flat, labels=range(4))
        cms[(arch, 'hierarchical', seed)] = confusion_matrix(y, pred_hier, labels=range(4))

results = pd.DataFrame(rows)
results.to_csv(TEST_OUT / 'test_results_per_seed.csv', index=False)

main_cols = ['accuracy', 'macro_f1', 'R_peaberry', 'R_premium', 'peaberry->premium',
             'premium->peaberry', 'other_errors', 'defect->premium', 'pair_share_of_errors']
pd.set_option('display.width', 220)
print(results[['arch', 'model', 'seed'] + main_cols].round(4).to_string(index=False))

stage_cols = ['stage1_acc', 'stage2_acc_on_pair', 'pair_lost_at_stage1', 'other_routed_to_stage2']
print('\nHierarchical details (error propagation):')
print(results[results['model'] == 'hierarchical'][['arch', 'seed'] + stage_cols].round(4).to_string(index=False))

## 7. Mean and standard deviation

In [ ]:
summary = results.groupby(['arch', 'model'])[main_cols].agg(['mean', 'std'])
summary.to_csv(TEST_OUT / 'test_results_summary.csv')
print(summary.round(4).to_string())

per_class = results.groupby(['arch', 'model'])[[f'{m}_{c}' for c in CLASSES for m in ['P', 'R', 'F1']]].mean()
per_class.to_csv(TEST_OUT / 'test_per_class_mean.csv')
print('\nPer-class mean (P / R / F1):')
print(per_class.round(4).T.to_string())

## 8. McNemar test

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

# Exact McNemar per seed: all 1,600 images and the 800 peaberry/premium images
tests = []
for arch in ARCHS:
    for seed in SEEDS:
        y, pf, ph = preds[(arch, seed)]
        for subset, mask in [('all_1600', np.ones_like(y, bool)), ('pair_800', np.isin(y, [2, 3]))]:
            cf, ch = (pf == y)[mask], (ph == y)[mask]
            # b: correct by flat only
            b = int((cf & ~ch).sum())
            # c: correct by two-stage only
            c = int((~cf & ch).sum())
            table = [[int((cf & ch).sum()), b],
                     [c, int((~cf & ~ch).sum())]]
            direction = 'hier_better' if c > b else ('flat_better' if b > c else 'equal')
            tests.append({'arch': arch, 'seed': seed, 'subset': subset,
                          'acc_flat': cf.mean(), 'acc_hier': ch.mean(),
                          'b_flat_only': b, 'c_hier_only': c, 'direction': direction,
                          'p_exact': mcnemar(table, exact=True).pvalue})

mc = pd.DataFrame(tests)
# Holm correction per architecture (6 tests each)
mc['p_holm'] = np.nan
for arch in ARCHS:
    idx = mc['arch'] == arch
    mc.loc[idx, 'p_holm'] = multipletests(mc.loc[idx, 'p_exact'], alpha=ALPHA, method='holm')[1]
mc['significant'] = mc['p_holm'] < ALPHA
mc.to_csv(TEST_OUT / 'mcnemar_results.csv', index=False)
print(mc.round(4).to_string(index=False))

## 9. Conclusion across seeds

In [ ]:
# Different only if significant in all three seeds, same direction
rows = []
for (arch, subset), g in mc.groupby(['arch', 'subset'], sort=False):
    n_hier = int((g['significant'] & (g['direction'] == 'hier_better')).sum())
    n_flat = int((g['significant'] & (g['direction'] == 'flat_better')).sum())
    if n_hier == len(SEEDS):
        verdict = 'different: hierarchical better (all seeds)'
    elif n_flat == len(SEEDS):
        verdict = 'different: flat better (all seeds)'
    elif n_hier + n_flat > 0:
        verdict = 'inconsistent across seeds'
    else:
        verdict = 'no significant difference'
    rows.append({'arch': arch, 'subset': subset,
                 'sig_hier_better': f'{n_hier}/{len(SEEDS)}',
                 'sig_flat_better': f'{n_flat}/{len(SEEDS)}',
                 'verdict': verdict})

conclusion = pd.DataFrame(rows)
conclusion.to_csv(TEST_OUT / 'mcnemar_conclusion.csv', index=False)
print(conclusion.to_string(index=False))

## 10. Confusion matrices

In [ ]:
import matplotlib.pyplot as plt

# Confusion matrices summed over three seeds
plt.rcParams.update({'font.family': 'serif',
                     'font.serif': ['Times New Roman', 'Liberation Serif', 'DejaVu Serif'],
                     'font.size': 9})
ARCH_NAMES = {'resnet50': 'ResNet-50', 'efficientnet_b0': 'EfficientNet-B0'}
APPROACH_NAMES = {'flat': 'Flat', 'hierarchical': 'Two-stage'}
n_max = len(SEEDS) * TEST_COUNT

fig = plt.figure(figsize=(6.3, 6.8), layout='constrained')
subfigs = fig.subfigures(2, 1, hspace=0.04)
letters = iter('abcd')
cm_rows = []
for subfig, arch in zip(subfigs, ARCHS):
    subfig.suptitle(ARCH_NAMES[arch], fontsize=10.5, fontweight='bold')
    axs = subfig.subplots(1, 2)
    for col, (ax, model_name) in enumerate(zip(axs, ['flat', 'hierarchical'])):
        m = sum(cms[(arch, model_name, s)] for s in SEEDS)
        assert (m.sum(1) == n_max).all()
        cm_rows += [[ARCH_NAMES[arch], APPROACH_NAMES[model_name], CLASSES[i]] + list(m[i]) for i in range(4)]
        ax.imshow(m, cmap='Blues', vmin=0, vmax=n_max)
        for r in range(4):
            for c in range(4):
                ax.text(c, r, str(m[r, c]), ha='center', va='center', fontsize=9.5,
                        color='white' if m[r, c] > n_max / 2 else 'black')
        ax.set_title(f'({next(letters)}) {APPROACH_NAMES[model_name]}', fontsize=10)
        ax.set_xticks(range(4)); ax.set_yticks(range(4))
        ax.set_xticklabels(CLASSES, rotation=30, ha='right')
        ax.set_yticklabels(CLASSES if col == 0 else [])
        ax.set_xlabel('Predicted class')
        if col == 0:
            ax.set_ylabel('True class')
        ax.tick_params(length=0)
        for s in ax.spines.values():
            s.set_linewidth(0.6)

fig.savefig(TEST_OUT / 'confusion_matrices_test.png', dpi=300)
plt.show()

# Save the summed matrices
cm_df = pd.DataFrame(cm_rows, columns=['architecture', 'approach', 'true_class'] + [f'pred_{c}' for c in CLASSES])
cm_df.to_csv(TEST_OUT / 'confusion_matrices_test.csv', index=False)
print(cm_df.to_string(index=False))